In [ ]:
import sys
from pathlib import Path
import os
import json

In [ ]:
# Adding project root (parent of notebooks/) to PYTHONPATH
sys.path.append(str(Path.cwd().parent))

In [ ]:
from ingestion.loaders import extract_all_texts
from ingestion.formatters import process_json_folder
from ingestion.resume_index import build_resume_index
from ingestion.job_index import build_jobs_index
from app.matching import search_index, read_txt
from agents.explainer_agent import explain_match_with_llm
from utils.json_utils import safe_json_parse
from agents.structuring_agent import process_resumes

In [ ]:
import urllib.request

try:
    urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
    print("✅ Ollama server is running (http://localhost:11434)")
except Exception as e:
    raise RuntimeError(
        "❌ Ollama is not reachable. Start Ollama and try again."
    ) from e


In [ ]:
extract_all_texts("../data/resume", "../data/resume_treated/resume_extract")

In [ ]:
process_resumes(
    input_folder="../data/resume_treated/resume_extract",
    output_folder="../data/resume_treated/resume_extract_json"
)

In [ ]:
sample = next(Path("../data/resume_treated/resume_extract_json").glob("*.json"))

with open(sample, "r", encoding="utf-8") as f:
    data = json.load(f)

data

In [ ]:
process_json_folder(
    "../data/resume_treated/resume_extract_json",
    "../data/resume_treated/resume_extract_text"
)

In [ ]:
BACKEND = "faiss"   # or "chroma"

result = build_resume_index(
    input_folder="../data/resume_treated/resume_extract_text",
    input_folder="../data/resume_treated/resume_extract_text",
    backend=BACKEND,
    faiss_index_path="../data/resume_treated/resume_index.faiss",
    mapping_path="../data/resume_treated/resume_index_mapping.json",
    chroma_dir="../data/resume_treated/chroma_resume_db",
    faiss_index_path="../data/resume_treated/resume_index.faiss",
    mapping_path="../data/resume_treated/resume_index_mapping.json",
    chroma_dir="../data/resume_treated/chroma_resume_db",
    chroma_collection="resumes",
)

result

# BUILD JOB INDEX

In [ ]:
result = build_jobs_index(
    input_folder="../data/job",
    faiss_index_path="../data/job_treated/jobs_index.faiss",
    mapping_path="../data/job_treated/jobs_index_mapping.json"
)
result

In [ ]:
job_text = read_txt("../data/job_test/sample_job.txt")

top_resumes = search_index(
    query_text=job_text,
    index_path="../data/resume_treated/resume_index.faiss",
    mapping_path="../data/resume_treated/resume_index_mapping.json",
    top_k=25
)
top_resumes

In [ ]:
client = build_llm_client() # type: ignore

def read_resume_text(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

for rank, hit in enumerate(top_resumes, start=1):
    resume_text = read_resume_text(
        filename=hit["filename"],
        base_dir="../data/resume_treated/resume_extract_text"
        base_dir="../data/resume_treated/resume_extract_text"
    )

    out = explain_match_with_llm(
        mode="job_to_resumes",
        similarity_score=hit["score"],
        job_text=job_text,
        resume_text=resume_text,
        top_k_rank=rank,
        client=client,
    )

    hit["llm_explanation"] = safe_json_parse(out["raw_json"])



In [ ]:
print("Score:", hit["score"])
print(hit["llm_explanation"]["explanation"])
print("Strengths:")
for s in hit["llm_explanation"]["strengths"]:
    print("-", s)
print("Gaps:")
for g in hit["llm_explanation"]["gaps"]:
    print("-", g)

In [ ]:
# =========================
# MODE 1 — JOB → TOP RESUMES
# =========================

import os
from agents.explainer_agent import explain_match_with_llm, build_llm_client
from utils.json_utils import safe_json_parse

client = build_llm_client()

def read_txt(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

RESUME_DIR = "../data/resume_treated/resume_extract_text"  # <- your folder with resume .txt
RESUME_DIR = "../data/resume_treated/resume_extract_text"  # <- your folder with resume .txt

for rank, hit in enumerate(top_resumes, start=1):
    resume_text = read_txt(hit["filename"], RESUME_DIR)

    out = explain_match_with_llm(
        mode="job_to_resumes",
        similarity_score=hit["score"],
        job_text=job_text,            # query = job
        resume_text=resume_text,      # candidate = resume
        top_k_rank=rank,
        client=client,
    )
    hit["llm_explanation"] = safe_json_parse(out["raw_json"])

    print("=" * 90)
    print(f"[JOB → RESUMES] Rank #{rank} | File: {hit['filename']} | Score: {hit['score']:.3f}\n")
    print(hit["llm_explanation"].get("explanation", ""))

    print("\nStrengths:")
    for s in hit["llm_explanation"].get("strengths", []):
        print("-", s)

    print("\nGaps:")
    for g in hit["llm_explanation"].get("gaps", []):
        print("-", g)


In [ ]:
resume_text = read_txt("../data/resume_treated/resume_extract_text/55.txt")
resume_text = read_txt("../data/resume_treated/resume_extract_text/55.txt")

top_jobs = search_index(
    query_text=resume_text,
    index_path="../data/jobs_index.faiss",
    mapping_path="../data/jobs_index_mapping.json",
    top_k=10
)
top_jobs


In [ ]:
# ======================
# MODE 2 — RESUME → TOP JOBS
# ======================

import os
from agents.explainer_agent import explain_match_with_llm, build_llm_client
from utils.json_utils import safe_json_parse

client = build_llm_client()

def read_txt(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

JOB_DIR = "../data/jobs"  # <- your folder with job .txt

for rank, hit in enumerate(top_jobs, start=1):
    job_candidate_text = read_txt(hit["filename"], JOB_DIR)

    out = explain_match_with_llm(
        mode="resume_to_jobs",
        similarity_score=hit["score"],
        resume_text=resume_text,           # query = resume
        job_text=job_candidate_text,       # candidate = job offer
        top_k_rank=rank,
        client=client,
    )
    hit["llm_explanation"] = safe_json_parse(out["raw_json"])

    print("=" * 90)
    print(f"[RESUME → JOBS] Rank #{rank} | File: {hit['filename']} | Score: {hit['score']:.3f}\n")
    print(hit["llm_explanation"].get("explanation", ""))

    print("\nStrengths:")
    for s in hit["llm_explanation"].get("strengths", []):
        print("-", s)

    print("\nGaps:")
    for g in hit["llm_explanation"].get("gaps", []):
        print("-", g)


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

def compatibility_score(job_text: str, resume_text: str, model_name="all-MiniLM-L6-v2") -> float:
    model = SentenceTransformer(model_name)
    emb = model.encode([job_text, resume_text], convert_to_numpy=True).astype("float32")
    emb = emb / np.clip(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12, None)
    return float(np.dot(emb[0], emb[1]))  # cosine

score = compatibility_score(
    read_txt("../data/jobs/sample_job.txt"),
    read_txt("../data/resume_treated/resume_extract_text/55.txt")
    read_txt("../data/resume_treated/resume_extract_text/55.txt")
)
score


In [ ]:
# =============================
# MODE 3 — PAIR COMPATIBILITY (RESUME ↔ JOB)
# =============================

import os
from agents.explainer_agent import explain_match_with_llm, build_llm_client
from utils.json_utils import safe_json_parse

client = build_llm_client()

def read_txt(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

RESUME_DIR = "../data/resume_treated/resume_extract_text"
RESUME_DIR = "../data/resume_treated/resume_extract_text"
JOB_DIR = "../data/jobs"

# Example inputs you choose (one resume + one job)
resume_filename = "55.txt"
job_filename = "sample_job.txt"

resume_pair_text = read_txt(resume_filename, RESUME_DIR)
job_pair_text = read_txt(job_filename, JOB_DIR)

# score must be computed before (cosine similarity between embeddings)
# score = ...
out = explain_match_with_llm(
    mode="pair_compatibility",
    similarity_score=score,
    resume_text=resume_pair_text,
    job_text=job_pair_text,
    client=client,
)

result = safe_json_parse(out["raw_json"])

print("=" * 90)
print(f"[PAIR COMPATIBILITY] Resume: {resume_filename} | Job: {job_filename} | Score: {score:.3f}\n")
print(result.get("explanation", ""))

print("\nStrengths:")
for s in result.get("strengths", []):
    print("-", s)

print("\nGaps:")
for g in result.get("gaps", []):
    print("-", g)

print("\nDecision:", result.get("decision", "N/A"))


In [ ]:
# =========================================
# MODE 3 — PAIR COMPATIBILITY + CV IMPROVEMENT
# =========================================

import os
from agents.explainer_agent import explain_match_with_llm, build_llm_client
from utils.json_utils import safe_json_parse

client = build_llm_client()

def read_txt(filename: str, base_dir: str) -> str:
    path = os.path.join(base_dir, filename)
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read().strip()

RESUME_DIR = "../data/resume_treated/resume_extract_text"
RESUME_DIR = "../data/resume_treated/resume_extract_text"
JOB_DIR = "../data/jobs"

# Selected pair
resume_filename = "55.txt"
job_filename = "sample_job.txt"

resume_text = read_txt(resume_filename, RESUME_DIR)
job_text = read_txt(job_filename, JOB_DIR)

# similarity_score must already be computed via embeddings
# score = cosine_similarity(...)
# Example:
# score = 0.78

out = explain_match_with_llm(
    mode="pair_compatibility_with_improvement",
    similarity_score=score,
    resume_text=resume_text,
    job_text=job_text,
    client=client,
)

result = safe_json_parse(out["raw_json"])

# --------------------
# DISPLAY
# --------------------
print("=" * 90)
print(
    f"[PAIR COMPATIBILITY]\n"
    f"Resume: {resume_filename}\n"
    f"Job: {job_filename}\n"
    f"Matching score: {score:.3f}\n"
)

print("Explanation:")
print(result.get("explanation", ""))

print("\nStrengths:")
for s in result.get("strengths", []):
    print("-", s)

print("\nGaps:")
for g in result.get("gaps", []):
    print("-", g)

print("\nCV Improvement Suggestions (targeted for this job):")
for i, rec in enumerate(result.get("cv_improvements", []), start=1):
    print(f"{i}. {rec}")

print("\nFinal Decision:", result.get("decision", "N/A"))


# LINKEDIN


In [ ]:
import pandas as pd

In [ ]:
df_job_posting = pd.read_csv("../data/linkedin_offers/linkedin_job_postings.csv")
df_skills = pd.read_csv("../data/linkedin_offers/job_skills.csv")
df_summary = pd.read_csv("../data/linkedin_offers/job_summary.csv")

In [ ]:
df_summary.dropna(inplace=True)
df_job_posting.dropna(inplace=True)
df_skills.dropna(inplace=True)

df_merged = df_job_posting.merge(df_skills, on="job_link").merge(df_summary, on="job_link")
df_merged.head()

In [ ]:
df_merged_reduced = df_merged[["job_link", "job_title", "company", "job_location", "search_city","job_type", "search_position", "job_skills", "job_summary"]]
df_merged_reduced.to_csv("linkedin_job_postings_cleaned.csv", index=False)

# LINKEDIN


In [ ]:
import pandas as pd

In [ ]:
df_job_posting = pd.read_csv("../data/linkedin_offers/linkedin_job_postings.csv")
df_skills = pd.read_csv("../data/linkedin_offers/job_skills.csv")
df_summary = pd.read_csv("../data/linkedin_offers/job_summary.csv")

In [ ]:
df_summary.dropna(inplace=True)
df_job_posting.dropna(inplace=True)
df_skills.dropna(inplace=True)

df_merged = df_job_posting.merge(df_skills, on="job_link").merge(df_summary, on="job_link")
df_merged.head()

In [ ]:
df_merged_reduced = df_merged[["job_link", "job_title", "company", "job_location", "search_city","job_type", "search_position", "job_skills", "job_summary"]]
df_merged_reduced.to_csv("linkedin_job_postings_cleaned.csv", index=False)

In [1]:
import os, sys, textwrap, json
from pathlib import Path

# ✅ Mets ici la racine du projet
PROJECT_ROOT = r"/home/lllm/Documents/SH/Smart_Resume_and_Job_Matcher"
os.chdir(PROJECT_ROOT)

# Si besoin (souvent inutile si tu es déjà à la racine)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("CWD =", os.getcwd())


CWD = /home/lllm/Documents/SH/Smart_Resume_and_Job_Matcher


In [2]:
from agents.explainer_agent import explain_match_with_llm
from agents.prompt_templates import format_messages
from app.pipeline import extract_text_from_file, sanitize_text_for_llm

In [3]:
import fitz
print("fitz module path:", fitz.__file__)
print("PyMuPDF version:", fitz.__doc__[:200])


fitz module path: /home/lllm/Documents/SH/.venv/lib/python3.12/site-packages/fitz/__init__.py
PyMuPDF version: PyMuPDF 1.26.7: Python bindings for the MuPDF 1.26.12 library (rebased implementation).
Python 3.12 running on linux (64-bit).



In [3]:
resume_path = Path("/home/lllm/Documents/SH/Smart_Resume_and_Job_Matcher/data/resume/10076271.pdf")
job_path = Path("/home/lllm/Documents/SH/Smart_Resume_and_Job_Matcher/data/job_test/sample_job.txt")

resume_file = {"filename": Path(resume_path).name, "bytes": Path(resume_path).read_bytes()}
job_file = {"filename": Path(job_path).name, "bytes": Path(job_path).read_bytes()}

resume_text = extract_text_from_file(resume_file)
job_text = extract_text_from_file(job_file)

resume_text = sanitize_text_for_llm(resume_text)
job_text = sanitize_text_for_llm(job_text)

print("resume chars:", len(resume_text))
print("job chars:", len(job_text))
print("\nRESUME preview:\n", resume_text[:400])
print("\nJOB preview:\n", job_text[:400])


resume chars: 5467
job chars: 1364

RESUME preview:
 CHIEF EXECUTIVE OFFICER
Summary
Award-winning executive and marketing professional experienced in high-volume, multi-unit, retail and business operations in the pharmaceutical,
financial services, and food and beverage industries. Demonstrated expertise in brand development, territory management, sales operations,
product launches, recruiting, and business development. Skilled in utilizing technol

JOB preview:
 We're hiring a Frontend Developer!

At our company, we're focused on developing cutting-edge B2E AI solutions, with a strong emphasis on asynchronous collaboration using GitHub and Slack, and weekly syncs.

**Who are we looking for?**
A frontend developer to spearhead the creation of a modern web application, handle the UI independently, and work closely with a small team to bring the entire produ


In [4]:
msgs = format_messages(
    mode="cv_job_fit",
    resume_text=resume_text,
    job_text=job_text,
    similarity_score=0.29,
    top_k_rank=None,
)

user_msg = msgs[1]["content"]
print("has JSON instruction:", "Return ONLY raw JSON" in user_msg)
print("\n=== USER last 600 ===\n")
print(user_msg[-600:])


has JSON instruction: True

=== USER last 600 ===

 Information Resources
Inc, Database, CUE, Quick Books

    Return ONLY raw JSON with EXACTLY these keys (no extra keys):
    {
    "explanation": "string (4-6 lines)",
    "strengths": ["string", "string", "string"],
    "gaps": ["string", "string", "string"],
    "decision": "strong_match|medium_match|weak_match",
    "advice": ["string", "string", "string"]
    }

    Rules:
    - Do NOT output keys like TEXT_A, TEXT_B, job_title, required_skills, ideal_candidate, compatibility, reasons.
    - Use ONLY info from the provided texts (no invention).
    - No markdown, no code block, only JSON.


In [6]:
def smart_trim(text: str, max_chars: int = 4500) -> str:
    if len(text) <= max_chars:
        return text
    head = text[:2500]
    tail = text[-2000:]
    return head + "\n\n[...TRUNCATED...]\n\n" + tail

resume_trim = smart_trim(resume_text, 4500)
job_trim = smart_trim(job_text, 3000)

out = explain_match_with_llm(
    mode="cv_job_fit",
    similarity_score=0.29,
    job_text=job_trim,
    resume_text=resume_trim,
    backend="OLLAMA",
)

print("parse_error:", out.get("parse_error"))
print("\n=== PARSED ===")
print(json.dumps(out.get("parsed"), indent=2, ensure_ascii=False))

print("\n=== RAW (first 1200) ===")
raw = out.get("raw_json") or ""
print(raw[:1200])



parse_error: None

=== PARSED ===
{
  "explanation": "The resume and job offer have a low cosine similarity score of 0.29, indicating that they are not highly compatible. The job offer is for a Frontend Developer position with expertise in React, TypeScript, and Tailwind CSS, while the resume highlights experience as a Chief Executive Officer with skills in brand development, project management, and sales operations.",
  "strengths": [
    "The candidate has experience working with teams and managing projects",
    "They have skills in utilizing technology to improve organizational efficiency",
    "Their background in marketing and sales could be relevant to the role of Frontend Developer"
  ],
  "gaps": [
    "Lack of direct experience as a Frontend Developer or with React, TypeScript, and Tailwind CSS",
    "No mention of asynchronous collaboration using GitHub and Slack, or weekly syncs",
    "No evidence of complex state management or API design experience"
  ],
  "decision": "wea

In [8]:
from pathlib import Path
from app.pipeline import run
import pprint


In [9]:
def load_file(path: str) -> dict:
    p = Path(path)
    return {
        "filename": p.name,
        "bytes": p.read_bytes()
    }

resume = load_file("/home/lllm/Documents/SH/Smart_Resume_and_Job_Matcher/data/resume/10219099.pdf")

jobs = [
    load_file("/home/lllm/Documents/SH/Smart_Resume_and_Job_Matcher/data/job_test/sample_job.txt"),
    load_file("/home/lllm/Documents/SH/Smart_Resume_and_Job_Matcher/data/job_test/sample_job2.txt"),
]


In [10]:
inputs = {
    "resume_file": resume,
    "job_offer_files": jobs,
    "add_explanations": True,     # IMPORTANT → déclenche Ollama
    "explain_top_n": 3,
}


In [11]:
out = run("cv_to_jobs", inputs)

print("STATUS:", out.get("status"))
print("MODE:", out.get("mode"))


STATUS: NOT_IMPLEMENTED_YET
MODE: cv_to_jobs
